In [ ]:
WORD_NUM = 15 # MNRead has 15 words at most per line

# Load in Data

In [10]:
import json
from pathlib import Path

output_dir = Path("../../filtered/mnread")
gt_dir = Path("../../data/mnread")
# Load the model output JSON (replace with your file or object)
with open(output_dir/ "gpt4o.json", "r") as f:
    model_output = json.load(f)

# Load the ground truth COCO JSON (annotations only or full dataset)
with open(gt_dir/"anno.json", "r") as f:
    ground_truth = json.load(f)
# Here we assume ground_truth has a key "annotations"
gt_annotations = ground_truth.get("annotations", [])
img_info = ground_truth.get("images", [])


In [19]:
# Create lookup dictionaries for ground truth annotations and image info by image_id.
# If there are multiple ground truth annotations per image, you might need to store a list.
gt_by_chart = {}
for info, ann in zip(img_info, gt_annotations):
    id = ann["id"]
    full = {}
    full["image_id"] = ann["image_id"]
    full["file_name"] = info["file_name"]
    full["Filter_no"] = info["Filter_no"]
    full["text"] = ann["caption"]
    gt_by_chart.setdefault(id, full)


In [ ]:
import ast
# Now iterate through the model outputs and match with ground truth and image info
full_out = {}
for output_item in model_output:
    image_id = output_item["image_id"]
    
    # Optionally, if your model output's "rec_texts" is a string representation of a list,
    # convert it to an actual list.
    rec_texts_str = output_item.get("rec_texts", "")
    try:
        rec_texts = ast.literal_eval(rec_texts_str)
    except Exception:
        rec_texts = rec_texts_str
    output_item["rec_texts"] = rec_texts

    # Retrieve corresponding ground truth annotations (if any)
    gt_matches = gt_by_chart.get(image_id, {})
    chart_no, reso, row_no = Path(gt_matches.get("file_name", "___")).stem.split("_")
    full_out[image_id] = gt_matches
    full_out[image_id]["rec_texts"] = rec_texts
    full_out[image_id]["chart_no"] = chart_no
    full_out[image_id]["reso"] = reso
    full_out[image_id]["row_no"] = row_no
    
    

In [51]:
Counter(pd.Series(['hello', 'world', 'world', 'bar', 'baz']).dropna()) & Counter(['x', 'world', 'bar', 'baz'])

Counter({'world': 1, 'bar': 1, 'baz': 1})

In [52]:
from collections import Counter
import pandas as pd

def count_matches_per_row(row):
    left_side = pd.Series(row['text'].lower().split(' '))
    right_side = pd.Series([i.lower() for i in row['rec_texts']])
    
    counter_left = Counter(left_side.dropna()) # dic
    counter_right = Counter(right_side.dropna())
    
    # 计算两侧相同元素的个数，确保每个元素只计算一次
    matches = sum((counter_left & counter_right).values())
    total = sum(counter_left.values())
    
    # char_level
    counter_left_char = Counter(''.join(left_side.dropna().tolist()))
    counter_right_char = Counter(''.join(right_side.dropna().tolist()))
    matches_char = sum((counter_left_char & counter_right_char).values())
    total_char = sum(counter_left_char.values())
    return [matches, total - matches, total_char-matches_char]


In [54]:
import pandas as pd
df = pd.DataFrame(full_out.values())
# df['text'].str.split(" ", expand=True)
# # Split the 'text' column into separate columns using space as the delimiter
# split_text = df['text'].str.split(" ", expand=True)

# # Rename the split columns to L1, L2, ..., Ln
# split_text.columns = [f"L{i+1}" for i in range(split_text.shape[1])]

# # Concatenate the new columns back to the original DataFrame
# df = pd.concat([df, split_text], axis=1)
df[['match','missing','missing_char']] = df.apply(lambda row: count_matches_per_row(row), axis=1).apply(pd.Series)
df

,image_id,file_name,Filter_no,text,rec_texts,chart_no,reso,row_no,match,missing,missing_char
0,0,Chart1_512_1.png,16,Very few people in this country will be happy ...,"[Very, few, people, in, this, country, will, b...",Chart1,512,1,12,0,0
1,1,Chart1_512_2.png,16,He went to the park to take a long walk and sa...,"[He, went, to, the, park, to, take, a, long, w...",Chart1,512,2,15,0,0
2,2,Chart1_512_3.png,16,In the end it did not matter that we were walk...,"[In, the, end, it, did, not, matter, that, we,...",Chart1,512,3,12,0,0
3,3,Chart1_512_4.png,16,The circus tent was put up in only three hours...,"[The, circus, tent, was, put, up, in, only, th...",Chart1,512,4,13,0,0
4,4,Chart1_512_5.png,16,We like to visit our local orchard when we nee...,"[We, like, to, visit, our, local, orchard, whe...",Chart1,512,5,12,0,0
...,...,...,...,...,...,...,...,...,...,...,...
795,795,Chart72_512_16.png,11,They need to read a book to your father before...,"[Vida, Boeing, Dendara, Solar, Eclipse, Collec...",Chart72,512,16,0,13,8
796,796,Chart72_512_17.png,11,The guards asked if their enemy put the bone u...,"[I, Don’t, Know]",Chart72,512,17,0,12,41
797,797,Chart72_512_18.png,11,Before skating they might like to speed to the...,"[Car, Sun, NUT]",Chart72,512,18,0,11,43
798,798,Chart72_512_19.png,11,Their friends hoped the member lost the ants u...,"[PASTASCIUTTA, ALFONSO, PANTALONI, LIMONATA, C...",Chart72,512,19,0,11,17
